<a href="https://colab.research.google.com/github/kelsbor/Prisma/blob/main/dados/prisma_coletor_docstrange_mcp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prisma · Coletor de extratos → Agente de armazenamento (MCP)

Este notebook lê extratos bancários em **vários formatos**, transforma tudo em transações padronizadas, confere a qualidade dos dados e grava no agente `agente-simples` pela ferramenta MCP `salvar_transacao`.

| Formato | Como é lido |
|---|---|
| PDF, PNG/JPG/TIFF/BMP/GIF, DOCX, PPTX, HTML | **DocStrange** (OCR + extração com esquema JSON) |
| CSV, TSV, TXT delimitado, XLSX/XLS, OFX | Leitura local, determinística (o DocStrange não aceita CSV, e não há por que enviar um arquivo já estruturado para fora) |
| PDF com texto, se o DocStrange falhar | Fallback local com `pypdf` |

**Fluxo:** upload → extração → normalização (datas, `R$ 1.384,62`, sinais) → checagens de qualidade e conciliação de saldo → consolidação entre arquivos → envio idempotente via MCP → verificação.

> ⚠️ **Privacidade / LGPD.** No modo `nuvem`, o DocStrange envia o arquivo para a API da Nanonets (fora do Brasil). Use **somente dados fictícios** nesse modo. Para dados reais, use o modo `gpu` (processa dentro do Colab) ou formatos lidos localmente (CSV, XLSX, OFX).

## 1. Instalação

In [69]:
# Instalação leve do DocStrange (modo nuvem).
# --no-deps evita que o pacote rebaixe o numpy do Colab e baixe torch/easyocr,
# que só são necessários no modo local com GPU.
!pip install -q docstrange --no-deps
!pip install -q requests pillow markdownify beautifulsoup4 lxml python-docx python-pptx openpyxl pypdf tqdm pdf2image pypandoc

# Modo local com GPU (dados não saem do Colab): use um runtime com GPU e rode
#   !pip install -q docstrange
# depois reinicie a sessão (Ambiente de execução → Reiniciar sessão) e use DOCSTRANGE_MODO = "gpu".

## 2. Configuração
Crie os secrets no ícone 🔑 do Colab: `MCP_TOKEN` (obrigatório) e `NANONETS_API_KEY` (para o modo nuvem; chave grátis em app.nanonets.com) e, opcionalmente, `GITHUB_TOKEN`. Sem a chave, o DocStrange tenta abrir um login no navegador e trava no Colab, por isso ela é exigida aqui.

Deixe `DRY_RUN = True` na primeira execução para ver tudo sem gravar.

In [70]:
import os

# ---------------- Configuração ----------------
URL_AGENTE = "https://agente-financeiro-i4qz.onrender.com"   # sem barra no final
TITULAR_ID = "000.000.000-00"                      # SEMPRE fictício em testes

DRY_RUN = False            # True = só mostra o que seria enviado. Mude para False para gravar.
DOCSTRANGE_MODO = "nuvem" # "nuvem" (API Nanonets) ou "gpu" (local, exige runtime com GPU)
TIMEOUT_S = 90            # o Render gratuito pode levar ~1 min para "acordar"

# Segredos: crie no ícone 🔑 do Colab e libere o acesso do notebook.
#   MCP_TOKEN         (obrigatório) token do agente de armazenamento
#   NANONETS_API_KEY  (obrigatório no modo "nuvem") chave grátis em https://app.nanonets.com/#/keys
def _segredo(nome):
    try:
        from google.colab import userdata
        return userdata.get(nome)
    except Exception:
        return os.environ.get(nome)

MCP_TOKEN = _segredo("MCP_TOKEN")
NANONETS_API_KEY = _segredo("NANONETS_API_KEY")

assert MCP_TOKEN, "Crie o secret MCP_TOKEN no Colab (ícone 🔑)."
if DOCSTRANGE_MODO == "nuvem" and not NANONETS_API_KEY:
    print("⚠️ Sem NANONETS_API_KEY: PDF/imagem/DOCX não serão lidos pelo DocStrange "
          "(sem a chave ele tenta abrir um login no navegador, que não funciona no Colab). "
          "CSV, TXT, XLSX e OFX continuam funcionando, pois são lidos localmente.")
print("Configuração ok · DRY_RUN =", DRY_RUN)

⚠️ Sem NANONETS_API_KEY: PDF/imagem/DOCX não serão lidos pelo DocStrange (sem a chave ele tenta abrir um login no navegador, que não funciona no Colab). CSV, TXT, XLSX e OFX continuam funcionando, pois são lidos localmente.
Configuração ok · DRY_RUN = False


## 3. Cliente MCP
Chamadas JSON-RPC ao endpoint `/mcp`, com novas tentativas para o *cold start* do Render.

In [71]:
import json, time, requests

class ErroMCP(Exception):
    pass

def chamar_ferramenta(nome, argumentos, tentativas=4):
    """Chama uma ferramenta do agente via MCP (JSON-RPC sobre HTTP, modo stateless)."""
    corpo = {"jsonrpc": "2.0", "id": 1, "method": "tools/call",
             "params": {"name": nome, "arguments": argumentos}}
    cabecalhos = {"Authorization": f"Bearer {MCP_TOKEN}",
                  "Content-Type": "application/json",
                  "Accept": "application/json, text/event-stream"}
    for i in range(tentativas):
        try:
            r = requests.post(f"{URL_AGENTE}/mcp", headers=cabecalhos, json=corpo, timeout=TIMEOUT_S)
        except requests.RequestException as e:
            erro = e
        else:
            if r.status_code == 401:
                raise ErroMCP("401: token recusado. Confira o secret MCP_TOKEN.")
            if r.status_code < 500:
                r.raise_for_status()
                resp = r.json()
                if "error" in resp:
                    raise ErroMCP(f"Erro JSON-RPC: {resp['error']}")
                res = resp["result"]
                texto = res["content"][0]["text"] if res.get("content") else "{}"
                if res.get("isError"):
                    raise ErroMCP(f"A ferramenta {nome} falhou: {texto}")
                return json.loads(texto)
            erro = f"HTTP {r.status_code}"
        espera = 5 * 2 ** i
        print(f"  ...tentativa {i+1} falhou ({erro}); nova tentativa em {espera}s")
        time.sleep(espera)
    raise ErroMCP(f"{nome}: sem resposta depois de {tentativas} tentativas")

print("health:", requests.get(f"{URL_AGENTE}/health", timeout=TIMEOUT_S).json())
existentes = chamar_ferramenta("consultar_titular", {"titular_id": TITULAR_ID})["registros"]
print(f"Conexão MCP ok · o titular já tem {len(existentes)} registro(s) no agente")

health: {'status': 'ok'}
Conexão MCP ok · o titular já tem 0 registro(s) no agente


## 4. Normalização
Converte valores (`-R$ 1.384,62`, `1384.62`, `(32,90)`, `32,90 D`) e datas (`2026-08-01`, `01/08/2026`, `01/08` com ano deduzido do período do extrato).

In [72]:
import re, unicodedata
from datetime import date, datetime
from decimal import Decimal, InvalidOperation

def _sem_acento(s):
    return "".join(c for c in unicodedata.normalize("NFKD", str(s)) if not unicodedata.combining(c))

def parse_valor(x):
    """'-R$ 1.384,62' · '1384.62' · '1.384,62-' · '(32,90)' · '32,90 D' · 3450 → Decimal com 2 casas."""
    if x is None:
        return None
    if isinstance(x, (int, float, Decimal)):
        if isinstance(x, float) and x != x:  # NaN
            return None
        return Decimal(str(x)).quantize(Decimal("0.01"))
    s = str(x).strip().upper().replace("\u2212", "-").replace("\xa0", " ")
    if not s:
        return None
    negativo = (s.startswith("-") or s.endswith("-") or (s.startswith("(") and s.endswith(")"))
                or bool(re.search(r"\s*D$", s)))
    s = re.sub(r"[^\d.,]", "", s)
    if not re.search(r"\d", s):
        return None
    if "," in s and "." in s:                       # o último separador é o decimal
        dec = "," if s.rfind(",") > s.rfind(".") else "."
        s = s.replace("." if dec == "," else ",", "").replace(dec, ".")
    elif "," in s:
        s = s.replace(".", "").replace(",", ".")
    elif re.fullmatch(r"\d{1,3}(\.\d{3})+", s):    # 1.384 = milhar
        s = s.replace(".", "")
    try:
        v = Decimal(s).quantize(Decimal("0.01"))
    except InvalidOperation:
        return None
    return -v if negativo else v

def parse_data(x, periodo=None):
    """Aceita AAAA-MM-DD, DD/MM/AAAA, DD/MM/AA, DD/MM (ano deduzido do período), AAAAMMDD (OFX)."""
    if x is None:
        return None
    if isinstance(x, datetime):
        return x.date()
    if isinstance(x, date):
        return x
    s = str(x).strip()
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d/%m/%y", "%d-%m-%Y", "%d.%m.%Y", "%Y%m%d"):
        try:
            return datetime.strptime(s[:10] if fmt != "%Y%m%d" else s[:8], fmt).date()
        except ValueError:
            pass
    m = re.fullmatch(r"(\d{1,2})/(\d{1,2})", s)
    if m:
        d, mes = int(m[1]), int(m[2])
        if periodo:                                  # escolhe o ano que cai dentro do período
            for ano in range(periodo[0].year, periodo[1].year + 1):
                try:
                    cand = date(ano, mes, d)
                except ValueError:
                    continue
                if periodo[0] <= cand <= periodo[1]:
                    return cand
        return None                                  # sem período não dá para saber o ano
    return None

def normalizar_descricao(s):
    return re.sub(r"\s+", " ", str(s or "")).strip().upper()

# --- metadados que aparecem no texto do extrato ---
RE_DATA = r"\d{2}/\d{2}/\d{4}"
RE_DINHEIRO = r"-?\s*\(?-?\s*(?:R\$)?\s*-?\d{1,3}(?:[.\s]?\d{3})*(?:[.,]\d{2})\)?-?(?:\s*[DC]\b)?"

def detectar_periodo(texto):
    m = re.search(rf"({RE_DATA})\s*(?:a|até|ate|-|–)\s*({RE_DATA})", texto or "", re.I)
    if m:
        return parse_data(m[1]), parse_data(m[2])
    return None

def detectar_saldo(texto, rotulo):
    m = re.search(rf"saldo\s+{rotulo}\W{{0,40}}?({RE_DINHEIRO})", _sem_acento(texto or ""), re.I)
    return parse_valor(m[1]) if m else None

## 5. Extratores por tipo de arquivo
Para PDF e imagens, o DocStrange recebe um **esquema JSON** do extrato. Se a resposta vier vazia, o notebook pede o markdown e lê a tabela; se ainda assim falhar e for PDF com texto, usa `pypdf`.

Detalhe importante: o DocStrange **não lança erro** quando a API falha, devolve conteúdo vazio. Por isso cada etapa é checada.

In [73]:
import csv, io
from pathlib import Path
import pandas as pd

# ---------- leitura local (estruturados: não precisam de OCR nem saem do Colab) ----------
APELIDOS = {
    "data":      ["data", "date", "dt", "data lancamento", "data movimento", "data mov"],
    "descricao": ["descricao", "historico", "lancamento", "memo", "description", "detalhe", "estabelecimento"],
    "valor":     ["valor", "amount", "valor r", "montante", "quantia"],
    "debito":    ["debito", "saida", "saidas", "debit"],
    "credito":   ["credito", "entrada", "entradas", "credit"],
    "saldo":     ["saldo", "balance"],
}

def _cab(s):
    return " ".join(re.sub(r"[^a-z0-9 ]", " ", _sem_acento(s).lower()).split())

def mapear_colunas(colunas):
    mapa = {}
    for campo, nomes in APELIDOS.items():
        for col in colunas:
            c = _cab(col) or ""
            if col in mapa.values():
                continue
            if c in nomes or any(c.startswith(n + " ") for n in nomes):
                mapa[campo] = col
                break
    ok = "data" in mapa and "descricao" in mapa and ("valor" in mapa or ("debito" in mapa and "credito" in mapa))
    return mapa if ok else None

def linhas_da_tabela(df, mapa, periodo=None):
    out = []
    for _, r in df.iterrows():
        d = parse_data(r[mapa["data"]], periodo)
        if "valor" in mapa:
            v = parse_valor(r[mapa["valor"]])
        else:
            deb, cre = parse_valor(r[mapa["debito"]]), parse_valor(r[mapa["credito"]])
            v = (cre or 0) - abs(deb or 0) if (deb is not None or cre is not None) else None
        if d is None or v is None:
            continue
        out.append({"data": d, "descricao": normalizar_descricao(r[mapa["descricao"]]), "valor": v,
                    "saldo": parse_valor(r[mapa["saldo"]]) if "saldo" in mapa else None})
    return out

def ler_delimitado(caminho):
    bruto = Path(caminho).read_bytes()
    for enc in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            texto = bruto.decode(enc); break
        except UnicodeDecodeError:
            continue
    try:
        sep = csv.Sniffer().sniff(texto[:4000], delimiters=";,\t|").delimiter
    except csv.Error:
        sep = ";"
    df = pd.read_csv(io.StringIO(texto), sep=sep, dtype=str, keep_default_na=False)
    mapa = mapear_colunas(df.columns)
    return (linhas_da_tabela(df, mapa) if mapa else []), texto

def ler_planilha(caminho):
    bruto = pd.read_excel(caminho, header=None, dtype=object)
    for i in range(min(30, len(bruto))):              # bancos costumam pôr cabeçalhos acima da tabela
        mapa = mapear_colunas([str(c) for c in bruto.iloc[i]])
        if mapa:
            df = bruto.iloc[i + 1:].copy()
            df.columns = [str(c) for c in bruto.iloc[i]]
            texto = "\n".join(" ".join(map(str, r)) for r in bruto.iloc[:i].values)
            return linhas_da_tabela(df, mapa, detectar_periodo(texto)), texto
    return [], ""

def ler_ofx(caminho):
    texto = Path(caminho).read_bytes().decode("latin-1")
    out = []
    for bloco in re.findall(r"<STMTTRN>(.*?)(?:</STMTTRN>|(?=<STMTTRN>)|</BANKTRANLIST>)", texto, re.S | re.I):
        campo = lambda t: (re.search(rf"<{t}>([^<\r\n]*)", bloco, re.I) or [None, None])[1]
        d, v = parse_data((campo("DTPOSTED") or "")[:8]), parse_valor(campo("TRNAMT"))
        if d and v is not None:
            out.append({"data": d, "descricao": normalizar_descricao(campo("MEMO") or campo("NAME")),
                        "valor": v, "saldo": None})
    return out, texto

# ---------- leitor de texto livre (markdown do DocStrange, texto de PDF, TXT sem cabeçalho) ----------
_RE_LINHA = re.compile(
    rf"^\s*(\d{{4}}-\d{{2}}-\d{{2}}|\d{{2}}/\d{{2}}(?:/\d{{2,4}})?)[\s,;]+(.+?)[\s,;]+({RE_DINHEIRO})"
    rf"(?:[\s,;]+({RE_DINHEIRO}))?\s*$")

def parse_texto(texto, periodo=None):
    periodo = periodo or detectar_periodo(texto)
    out = []
    for linha in (texto or "").splitlines():
        if "|" in linha:                                   # linha de tabela markdown
            cel = [c.strip() for c in linha.strip().strip("|").split("|")]
            idx = next((i for i, c in enumerate(cel) if parse_data(c, periodo)), None)
            if idx is None:
                continue
            resto = cel[idx + 1:]
            din = [i for i, c in enumerate(resto) if re.fullmatch(RE_DINHEIRO, c) and parse_valor(c) is not None]
            if not din:
                continue
            desc = " ".join(resto[:din[0]])
            valor, saldo = parse_valor(resto[din[0]]), (parse_valor(resto[din[1]]) if len(din) > 1 else None)
            d = parse_data(cel[idx], periodo)
        else:
            m = _RE_LINHA.match(linha)
            if not m:
                continue
            d, desc = parse_data(m[1], periodo), m[2].strip(" ,;")
            valor, saldo = parse_valor(m[3]), parse_valor(m[4]) if m[4] else None
        if d and valor is not None and desc:
            out.append({"data": d, "descricao": normalizar_descricao(desc), "valor": valor, "saldo": saldo})
    return out

# ---------- DocStrange (PDF, imagens, Word, PowerPoint, HTML) ----------
ESQUEMA_EXTRATO = {
    "banco": "string",
    "periodo_inicio": "string (DD/MM/AAAA)",
    "periodo_fim": "string (DD/MM/AAAA)",
    "saldo_inicial": "number",
    "saldo_final": "number",
    "transacoes": [{"data": "string (DD/MM/AAAA)", "descricao": "string",
                    "valor": "number (negativo = saída)", "saldo": "number"}],
}
_extrator = None

def _docstrange():
    global _extrator
    if _extrator is None:
        from docstrange import DocumentExtractor
        if DOCSTRANGE_MODO == "gpu":
            _extrator = DocumentExtractor(gpu=True)
        else:
            if not NANONETS_API_KEY:
                raise RuntimeError("Defina o secret NANONETS_API_KEY para usar o DocStrange na nuvem.")
            _extrator = DocumentExtractor(api_key=NANONETS_API_KEY)
    return _extrator

def _achar_transacoes(obj):
    """Procura, em qualquer nível do JSON devolvido, a lista de lançamentos."""
    if isinstance(obj, list) and obj and all(isinstance(i, dict) for i in obj) \
            and any("valor" in i or "amount" in i for i in obj):
        return obj
    if isinstance(obj, dict):
        for v in obj.values():
            achou = _achar_transacoes(v)
            if achou:
                return achou
    return None

def ler_docstrange(caminho):
    res = _docstrange().extract(str(caminho))
    # O DocStrange não lança erro quando a API falha: devolve conteúdo vazio. Por isso checamos tudo.
    dados = res.extract_data(json_schema=ESQUEMA_EXTRATO)
    meta = dados.get("structured_data") if dados.get("format") == "structured_json" else None
    if isinstance(meta, str):
        try: meta = json.loads(meta)
        except ValueError: meta = None
    meta = meta if isinstance(meta, dict) else {}
    periodo = None
    if meta.get("periodo_inicio") and meta.get("periodo_fim"):
        periodo = (parse_data(meta["periodo_inicio"]), parse_data(meta["periodo_fim"]))
        periodo = periodo if all(periodo) else None
    brutas = _achar_transacoes(meta) or []
    texto = ""
    if not brutas or (periodo is None and any(re.fullmatch(r"\d{1,2}/\d{1,2}", str(t.get("data", "")).strip()) for t in brutas)):
        texto = res.extract_markdown() or ""
        periodo = periodo or detectar_periodo(texto)
    linhas = []
    for t in brutas:
        d, v = parse_data(t.get("data"), periodo), parse_valor(t.get("valor", t.get("amount")))
        if d and v is not None:
            linhas.append({"data": d, "descricao": normalizar_descricao(t.get("descricao", t.get("description"))),
                           "valor": v, "saldo": parse_valor(t.get("saldo"))})
    metodo = "docstrange · json_schema"
    if not linhas and texto:
        linhas, metodo = parse_texto(texto, periodo), "docstrange · markdown"
    extra = {"periodo": periodo,
             "saldo_inicial": parse_valor(meta.get("saldo_inicial")) if meta.get("saldo_inicial") is not None else detectar_saldo(texto, "inicial"),
             "saldo_final": parse_valor(meta.get("saldo_final")) if meta.get("saldo_final") is not None else detectar_saldo(texto, "final")}
    return linhas, texto, metodo, extra

def ler_pdf_local(caminho):
    from pypdf import PdfReader
    texto = "\n".join(p.extract_text() or "" for p in PdfReader(str(caminho)).pages)
    return parse_texto(texto), texto

# ---------- roteador por tipo de arquivo ----------
LOCAIS = {".csv", ".tsv", ".txt", ".xlsx", ".xls", ".ofx"}
VIA_DOCSTRANGE = {".pdf", ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".gif",
                  ".docx", ".doc", ".pptx", ".ppt", ".html", ".htm"}

def extrair(caminho):
    p = Path(caminho); ext = p.suffix.lower()
    r = {"arquivo": p.name, "tipo": ext.lstrip("."), "metodo": None, "transacoes": [],
         "periodo": None, "saldo_inicial": None, "saldo_final": None, "avisos": []}
    texto = ""
    try:
        if ext in (".csv", ".tsv", ".txt"):
            r["transacoes"], texto = ler_delimitado(p); r["metodo"] = "local · delimitado"
            if not r["transacoes"]:
                r["transacoes"], r["metodo"] = parse_texto(texto), "local · texto livre"
        elif ext in (".xlsx", ".xls"):
            r["transacoes"], texto = ler_planilha(p); r["metodo"] = "local · planilha"
        elif ext == ".ofx":
            r["transacoes"], texto = ler_ofx(p); r["metodo"] = "local · OFX"

        if not r["transacoes"] and (ext in VIA_DOCSTRANGE or ext in (".txt", ".xlsx", ".xls")):
            try:
                r["transacoes"], texto, r["metodo"], extra = ler_docstrange(p)
                r.update({k: v for k, v in extra.items() if v is not None})
            except Exception as e:
                r["avisos"].append(f"DocStrange falhou: {e}")
            if not r["transacoes"] and ext == ".pdf":
                r["transacoes"], texto = ler_pdf_local(p)
                r["metodo"] = "fallback · texto do PDF (pypdf)"
                r["avisos"].append("PDF lido sem DocStrange; só funciona se o PDF tiver camada de texto.")
        elif ext not in LOCAIS | VIA_DOCSTRANGE:
            r["avisos"].append(f"Tipo .{ext} não suportado.")
    except Exception as e:
        r["avisos"].append(f"Erro ao ler: {e}")

    r["periodo"] = r["periodo"] or detectar_periodo(texto)
    r["saldo_inicial"] = r["saldo_inicial"] if r["saldo_inicial"] is not None else detectar_saldo(texto, "inicial")
    r["saldo_final"] = r["saldo_final"] if r["saldo_final"] is not None else detectar_saldo(texto, "final")
    if not r["transacoes"]:
        r["avisos"].append("Nenhuma transação encontrada.")
    return r

## 6. Extratos de entrada
Por padrão (`FONTE = "github"`), baixa todos os arquivos de [`agente-simples/dados`](https://github.com/kelsbor/Prisma/tree/main/agente-simples/dados) no repositório. Arquivo novo colocado nessa pasta entra automaticamente na próxima execução.

A listagem usa a API do GitHub, que no Colab costuma bater no limite de 60 chamadas/hora por IP. Nesse caso o notebook usa a lista de arquivos conhecida, ou você pode criar o secret opcional `GITHUB_TOKEN`. Para enviar arquivos do seu computador, use `FONTE = "upload"`.

In [74]:
from pathlib import Path
import requests

# De onde vêm os extratos: "github" (pasta agente-simples/dados do repositório) ou "upload" (do seu computador)
FONTE = "github"
REPO, BRANCH, PASTA_REPO = "kelsbor/Prisma", "main", "agente-simples/dados"
# Lista de reserva: usada se a API do GitHub recusar (limite de 60 chamadas/hora por IP, comum no Colab)
ARQUIVOS_CONHECIDOS = ["extrato_ficticio_csv.pdf", "extrato_ficticio_prisma.png", "extrato_limpo_excel_csv.txt"]

PASTA = Path("/content/extratos") if Path("/content").exists() else Path("extratos")
PASTA.mkdir(parents=True, exist_ok=True)

def listar_pasta_github():
    url = f"https://api.github.com/repos/{REPO}/contents/{PASTA_REPO}?ref={BRANCH}"
    token = _segredo("GITHUB_TOKEN")                     # opcional; aumenta o limite da API
    cab = {"Authorization": f"Bearer {token}"} if token else {}
    try:
        r = requests.get(url, headers=cab, timeout=30)
        if r.ok and isinstance(r.json(), list):
            return [i["name"] for i in r.json() if i["type"] == "file"]
        print(f"  API do GitHub indisponível (HTTP {r.status_code}); usando a lista conhecida")
    except requests.RequestException as e:
        print(f"  API do GitHub indisponível ({e}); usando a lista conhecida")
    return ARQUIVOS_CONHECIDOS

if FONTE == "github":
    for p in PASTA.iterdir():                            # evita misturar com arquivos de execuções anteriores
        if p.is_file():
            p.unlink()
    for nome in listar_pasta_github():
        raw = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/{PASTA_REPO}/{nome}"
        r = requests.get(raw, timeout=60)
        if r.ok:
            (PASTA / nome).write_bytes(r.content)
            print(f"  ⬇️ {nome} ({len(r.content):,} bytes)")
        else:
            print(f"  ❌ {nome}: HTTP {r.status_code}")
else:
    try:
        from google.colab import files
        for nome, conteudo in files.upload().items():   # selecione um ou vários arquivos
            (PASTA / nome).write_bytes(conteudo)
    except ImportError:
        print("Fora do Colab: copie os arquivos para", PASTA)

arquivos = sorted(p for p in PASTA.iterdir() if p.is_file())
print(f"\n{len(arquivos)} arquivo(s) para processar:", *[p.name for p in arquivos], sep="\n  ")

  ⬇️ extrato_ficticio_csv.pdf (26,864 bytes)
  ⬇️ extrato_ficticio_prisma.png (1,089,000 bytes)
  ⬇️ extrato_limpo_excel_csv.txt (3,319 bytes)

3 arquivo(s) para processar:
  extrato_ficticio_csv.pdf
  extrato_ficticio_prisma.png
  extrato_limpo_excel_csv.txt


## 7. Extração

In [75]:
resultados = []
for p in arquivos:
    print(f"→ {p.name}")
    r = extrair(p)
    resultados.append(r)
    print(f"   {len(r['transacoes'])} transações · método: {r['metodo']}")
    for a in r["avisos"]:
        print("   ⚠️", a)

df_bruto = pd.DataFrame([{**t, "arquivo": r["arquivo"], "tipo": r["tipo"], "metodo": r["metodo"]}
                         for r in resultados for t in r["transacoes"]])
df_bruto.head(10)

→ extrato_ficticio_csv.pdf
   79 transações · método: fallback · texto do PDF (pypdf)
   ⚠️ DocStrange falhou: Defina o secret NANONETS_API_KEY para usar o DocStrange na nuvem.
   ⚠️ PDF lido sem DocStrange; só funciona se o PDF tiver camada de texto.
→ extrato_ficticio_prisma.png
   0 transações · método: None
   ⚠️ DocStrange falhou: Defina o secret NANONETS_API_KEY para usar o DocStrange na nuvem.
   ⚠️ Nenhuma transação encontrada.
→ extrato_limpo_excel_csv.txt
   77 transações · método: local · delimitado


,data,descricao,valor,saldo,arquivo,tipo,metodo
0,2026-08-01,TARIFA PACOTE SERVICOS,-32.90,None,extrato_ficticio_csv.pdf,pdf,fallback · texto do PDF (pypdf)
1,2026-08-03,COMPRA CARTAO DEB SUPERMERC BOM PRECO,-187.43,None,extrato_ficticio_csv.pdf,pdf,fallback · texto do PDF (pypdf)
2,2026-08-04,PAG*IFOOD,-42.90,None,extrato_ficticio_csv.pdf,pdf,fallback · texto do PDF (pypdf)
3,2026-08-05,DL*UBERRIDES,-18.75,None,extrato_ficticio_csv.pdf,pdf,fallback · texto do PDF (pypdf)
4,2026-08-06,CRED SALARIO EMPRESA LTDA,3450.00,None,extrato_ficticio_csv.pdf,pdf,fallback · texto do PDF (pypdf)
5,2026-08-06,PAGTO FATURA CARTAO FINAL 0000,-1384.62,None,extrato_ficticio_csv.pdf,pdf,fallback · texto do PDF (pypdf)
6,2026-08-07,DEB AUT ENERGIA DISTRIB,-148.30,None,extrato_ficticio_csv.pdf,pdf,fallback · texto do PDF (pypdf)
7,2026-08-08,NETFLIX.COM,-44.90,None,extrato_ficticio_csv.pdf,pdf,fallback · texto do PDF (pypdf)
8,2026-08-08,SPOTIFY P1A2B3C4,-21.90,None,extrato_ficticio_csv.pdf,pdf,fallback · texto do PDF (pypdf)
9,2026-08-10,PIX ENVIADO - IMOBILIARIA LAR,-950.00,None,extrato_ficticio_csv.pdf,pdf,fallback · texto do PDF (pypdf)


## 8. Qualidade e conciliação
Nada é corrigido automaticamente; os problemas viram **alertas** gravados junto da transação (solução explicável):

- `repetido_no_mesmo_dia`: mesmo lançamento duas vezes. É mantido, porque pode ser uma cobrança dupla real (ex.: depois aparece o estorno).
- `valor_irrisorio`: ex.: PIX de R$ 0,01 de teste.
- `descricao_parecida_com:X`: possível erro de digitação (ex.: `PAG*IFOD`).
- `estorna:<data>`: estorno ligado ao débito original.
- Conciliação: saldo inicial + movimentos = saldo final, e saldo linha a linha quando o extrato traz a coluna de saldo.

In [76]:
import hashlib, difflib
from collections import Counter, defaultdict

def chave(t):
    """Identidade da transação, independente da descrição (que muda entre PDF, imagem e CSV)."""
    return hashlib.sha256(f"{t['data']}|{t['valor']}|{t['ocorrencia']}".encode()).hexdigest()[:16]

def checar_arquivo(r):
    ts = sorted(r["transacoes"], key=lambda t: t["data"])  # sort estável: mantém a ordem do extrato no mesmo dia
    cont = Counter()
    for t in ts:
        cont[(t["data"], t["valor"])] += 1
        t["ocorrencia"] = cont[(t["data"], t["valor"])]
        t["chave"] = chave(t)
        t["alertas"] = []

    # 1) lançamento idêntico repetido no mesmo arquivo: mantém (pode ser cobrança dupla real) e avisa
    iguais = Counter((t["data"], t["descricao"], t["valor"]) for t in ts)
    for t in ts:
        if iguais[(t["data"], t["descricao"], t["valor"])] > 1:
            t["alertas"].append("repetido_no_mesmo_dia")
    # 2) valor irrisório (ex.: PIX de teste de R$ 0,01)
    for t in ts:
        if abs(t["valor"]) < 1:
            t["alertas"].append("valor_irrisorio")
    # 3) descrição quase igual a outra mais frequente (possível erro de digitação, ex.: PAG*IFOD)
    freq = Counter(t["descricao"] for t in ts)
    comuns = [d for d, n in freq.items() if n >= 2]
    for t in ts:
        if freq[t["descricao"]] == 1:
            par = difflib.get_close_matches(t["descricao"], comuns, n=1, cutoff=0.85)
            if par:
                t["alertas"].append(f"descricao_parecida_com:{par[0]}")
    # 4) estornos: liga ao débito de mesmo valor anterior
    for t in ts:
        if t["valor"] > 0 and "ESTORNO" in t["descricao"]:
            orig = [o for o in ts if o["valor"] == -t["valor"] and o["data"] <= t["data"] and o is not t]
            if orig:
                t["alertas"].append(f"estorna:{orig[-1]['data']}")
    # 5) conciliação de saldos
    r["conciliacao"] = []
    if r["saldo_inicial"] is not None and r["saldo_final"] is not None:
        calc = r["saldo_inicial"] + sum(t["valor"] for t in ts)
        ok = calc == r["saldo_final"]
        r["conciliacao"].append(f"saldo inicial + movimentos = {calc} · extrato diz {r['saldo_final']} → "
                                + ("✅ confere" if ok else f"❌ diferença de {r['saldo_final'] - calc}"))
    if ts and all(t.get("saldo") is not None for t in ts):
        anterior = r["saldo_inicial"] if r["saldo_inicial"] is not None else ts[0]["saldo"] - ts[0]["valor"]
        quebras = 0
        for t in ts:
            if anterior + t["valor"] != t["saldo"]:
                t["alertas"].append("saldo_nao_confere"); quebras += 1
            anterior = t["saldo"]
        r["conciliacao"].append("saldo linha a linha: " + ("✅ confere" if not quebras else f"❌ {quebras} linha(s) não batem"))
    if not r["conciliacao"]:
        r["conciliacao"].append("sem saldos no arquivo: não dá para conciliar")
    r["transacoes"] = ts
    return r

for r in resultados:
    checar_arquivo(r)
    entradas = sum(t["valor"] for t in r["transacoes"] if t["valor"] > 0)
    saidas = sum(t["valor"] for t in r["transacoes"] if t["valor"] < 0)
    print(f"\n📄 {r['arquivo']} ({r['metodo']})")
    print(f"   {len(r['transacoes'])} transações · entradas {entradas} · saídas {saidas} · líquido {entradas + saidas}")
    if r["periodo"]:
        print(f"   período {r['periodo'][0]:%d/%m/%Y} a {r['periodo'][1]:%d/%m/%Y}")
    for c in r["conciliacao"]:
        print("  ", c)
    for t in r["transacoes"]:
        if t["alertas"]:
            print(f"   ⚠️ {t['data']:%d/%m} {t['descricao'][:32]:<32} {t['valor']:>10} → {', '.join(t['alertas'])}")


📄 extrato_ficticio_csv.pdf (fallback · texto do PDF (pypdf))
   79 transações · entradas 12078.60 · saídas -12420.04 · líquido -341.44
   sem saldos no arquivo: não dá para conciliar
   ⚠️ 19/08 PAG*IFOD                             -29.90 → descricao_parecida_com:PAG*IFOOD
   ⚠️ 03/09 PAG*IFOOD                            -48.70 → repetido_no_mesmo_dia
   ⚠️ 03/09 PAG*IFOOD                            -48.70 → repetido_no_mesmo_dia
   ⚠️ 04/09 PIX ENVIADO - J S                     -0.01 → valor_irrisorio
   ⚠️ 14/09 ESTORNO PAG*IFOOD                     48.70 → estorna:2026-09-03
   ⚠️ 17/09 ESTORNO COMPRA LOJA ELETRO           349.90 → estorna:2026-09-10

📄 extrato_ficticio_prisma.png (None)
   0 transações · entradas 0 · saídas 0 · líquido 0
   sem saldos no arquivo: não dá para conciliar

📄 extrato_limpo_excel_csv.txt (local · delimitado)
   77 transações · entradas 12029.90 · saídas -12371.34 · líquido -341.44
   sem saldos no arquivo: não dá para conciliar
   ⚠️ 19/08 PAG*IFOD     

## 9. Consolidação entre arquivos
Assume que os arquivos são da **mesma conta**. A chave de cada transação é `data + valor + ordem no dia`, e não a descrição, porque a descrição muda entre fontes. Quando um arquivo tem lançamentos que outro não tem, isso aparece aqui. A descrição gravada vem da fonte mais confiável (CSV/OFX > planilha/TXT > PDF > imagem), e as outras ficam em `descricoes_alternativas`.

In [77]:
# Consolida todos os arquivos (assume que são extratos da MESMA conta).
# A chave usa data + valor + ordem no dia, porque a descrição varia entre fontes
# (ex.: "DL*UBERRIDES" no CSV e "UBER* TRIP HELP.UBER.COM" na imagem).
PRIORIDADE = {"csv": 0, "tsv": 0, "ofx": 0, "xlsx": 1, "xls": 1, "txt": 1, "pdf": 2}  # demais (imagem etc.) = 3

por_chave = defaultdict(list)
for r in resultados:
    for t in r["transacoes"]:
        por_chave[t["chave"]].append((r, t))

consolidado = []
for k, ocorr in por_chave.items():
    ocorr.sort(key=lambda rt: PRIORIDADE.get(rt[0]["tipo"], 3))
    r0, t0 = ocorr[0]
    descricoes = sorted({t["descricao"] for _, t in ocorr} - {t0["descricao"]})
    alertas = sorted({a for _, t in ocorr for a in t["alertas"] if a != "saldo_nao_confere"})
    consolidado.append({
        "data": t0["data"].isoformat(), "descricao": t0["descricao"], "valor": float(t0["valor"]),
        "moeda": "BRL", "chave": k,
        "fontes": [{"arquivo": r["arquivo"], "tipo": r["tipo"], "metodo": r["metodo"]} for r, _ in ocorr],
        **({"descricoes_alternativas": descricoes} if descricoes else {}),
        **({"alertas": alertas} if alertas else {}),
    })
consolidado.sort(key=lambda d: (d["data"], d["chave"]))

# Diferenças entre fontes: transações que aparecem em uns arquivos e não em outros
nomes = [r["arquivo"] for r in resultados if r["transacoes"]]
faltas = [(d, sorted(set(nomes) - {f["arquivo"] for f in d["fontes"]})) for d in consolidado]
faltas = [(d, f) for d, f in faltas if f]
print(f"{len(consolidado)} transações únicas a partir de {len(nomes)} arquivo(s)")
if len(nomes) > 1:
    print(f"{len(faltas)} transação(ões) não aparecem em todos os arquivos:")
    for d, f in faltas:
        print(f"   {d['data']} {d['descricao'][:30]:<30} {d['valor']:>10.2f}  ausente em: {', '.join(f)}")

pd.DataFrame(consolidado)[["data", "descricao", "valor", "alertas"] if any("alertas" in d for d in consolidado)
                          else ["data", "descricao", "valor"]]

79 transações únicas a partir de 2 arquivo(s)
2 transação(ões) não aparecem em todos os arquivos:
   2026-09-03 PAG*IFOOD                          -48.70  ausente em: extrato_limpo_excel_csv.txt
   2026-09-14 ESTORNO PAG*IFOOD                   48.70  ausente em: extrato_limpo_excel_csv.txt


,data,descricao,valor,alertas
0,2026-08-01,TARIFA PACOTE SERVICOS,-32.90,NaN
1,2026-08-03,COMPRA CARTAO DEB SUPERMERC BOM PRECO,-187.43,NaN
2,2026-08-04,PAG*IFOOD,-42.90,NaN
3,2026-08-05,DL*UBERRIDES,-18.75,NaN
4,2026-08-06,PAGTO FATURA CARTAO FINAL 0000,-1384.62,NaN
...,...,...,...,...
74,2026-10-20,SAQUE 24H BANCO24HORAS,-200.00,NaN
75,2026-10-22,COMPRA CARTAO DEB SUPERMERC BOM PRECO,-176.94,NaN
76,2026-10-24,PIX ENVIADO - ACADEMIA CORPO FIT,-99.90,NaN
77,2026-10-25,RECARGA CELULAR OPERADORA,-30.00,NaN


## 10. Envio ao agente
O agente não deduplica, então o notebook consulta o que já existe (`consultar_titular`) e envia só chaves novas. Rodar duas vezes não duplica nada.

In [78]:
from datetime import timezone

# Idempotência: o agente aceita qualquer JSON e não deduplica,
# então consultamos o que já existe e só enviamos chaves novas.
ja_gravadas = {reg["dados"].get("chave") for reg in
               chamar_ferramenta("consultar_titular", {"titular_id": TITULAR_ID})["registros"]}
novas = [d for d in consolidado if d["chave"] not in ja_gravadas]
print(f"{len(consolidado) - len(novas)} já estão no agente · {len(novas)} novas")

if DRY_RUN:
    print("\nDRY_RUN = True → nada foi gravado. Exemplo do JSON que seria enviado:")
    print(json.dumps(novas[:2], ensure_ascii=False, indent=2))
else:
    agora = datetime.now(timezone.utc).isoformat(timespec="seconds")
    log = []
    for i, d in enumerate(novas, 1):
        resp = chamar_ferramenta("salvar_transacao",
                                 {"titular_id": TITULAR_ID, "dados": {**d, "ingerido_em": agora,
                                                                      "coletor": "colab-docstrange-v1"}})
        log.append({"chave": d["chave"], "id": resp.get("id")})
        if i % 10 == 0 or i == len(novas):
            print(f"  {i}/{len(novas)} gravadas")
    print("✅ envio concluído")

0 já estão no agente · 79 novas
  10/79 gravadas
  20/79 gravadas
  30/79 gravadas
  40/79 gravadas
  50/79 gravadas
  60/79 gravadas
  70/79 gravadas
  79/79 gravadas
✅ envio concluído


## 11. Verificação

In [79]:

import json, requests
import pandas as pd
from google.colab import userdata

URL_AGENTE = "https://agente-financeiro-i4qz.onrender.com"
TITULAR_ID = "000.000.000-00"

r = requests.post(f"{URL_AGENTE}/mcp", timeout=90,
                  headers={"Authorization": f"Bearer {userdata.get('MCP_TOKEN')}",
                           "Content-Type": "application/json",
                           "Accept": "application/json, text/event-stream"},
                  json={"jsonrpc": "2.0", "id": 1, "method": "tools/call",
                        "params": {"name": "consultar_titular",
                                   "arguments": {"titular_id": TITULAR_ID}}})

registros = json.loads(r.json()["result"]["content"][0]["text"])["registros"]
df = pd.DataFrame([reg["dados"] for reg in registros])
print(len(df), "registros")
df

79 registros


,data,descricao,valor,moeda,chave,fontes,ingerido_em,coletor,alertas
0,2026-08-01,TARIFA PACOTE SERVICOS,-32.90,BRL,299f28be430345fe,"[{'arquivo': 'extrato_limpo_excel_csv.txt', 't...",2026-10-29T21:18:20+00:00,colab-docstrange-v1,NaN
1,2026-08-03,COMPRA CARTAO DEB SUPERMERC BOM PRECO,-187.43,BRL,3ac09c3d91560eea,"[{'arquivo': 'extrato_limpo_excel_csv.txt', 't...",2026-10-29T21:18:20+00:00,colab-docstrange-v1,NaN
2,2026-08-04,PAG*IFOOD,-42.90,BRL,eb0d6eac1eef1d4e,"[{'arquivo': 'extrato_limpo_excel_csv.txt', 't...",2026-10-29T21:18:20+00:00,colab-docstrange-v1,NaN
3,2026-08-05,DL*UBERRIDES,-18.75,BRL,b0920008dd126f74,"[{'arquivo': 'extrato_limpo_excel_csv.txt', 't...",2026-10-29T21:18:20+00:00,colab-docstrange-v1,NaN
4,2026-08-06,PAGTO FATURA CARTAO FINAL 0000,-1384.62,BRL,51951a386de5ad5a,"[{'arquivo': 'extrato_limpo_excel_csv.txt', 't...",2026-10-29T21:18:20+00:00,colab-docstrange-v1,NaN
...,...,...,...,...,...,...,...,...,...
74,2026-10-20,SAQUE 24H BANCO24HORAS,-200.00,BRL,780e93d8de633681,"[{'arquivo': 'extrato_limpo_excel_csv.txt', 't...",2026-10-29T21:18:20+00:00,colab-docstrange-v1,NaN
75,2026-10-22,COMPRA CARTAO DEB SUPERMERC BOM PRECO,-176.94,BRL,f8b873f18243b311,"[{'arquivo': 'extrato_limpo_excel_csv.txt', 't...",2026-10-29T21:18:20+00:00,colab-docstrange-v1,NaN
76,2026-10-24,PIX ENVIADO - ACADEMIA CORPO FIT,-99.90,BRL,39fc9e31329c8d8b,"[{'arquivo': 'extrato_limpo_excel_csv.txt', 't...",2026-10-29T21:18:20+00:00,colab-docstrange-v1,NaN
77,2026-10-25,RECARGA CELULAR OPERADORA,-30.00,BRL,89711cc887d9dc1e,"[{'arquivo': 'extrato_limpo_excel_csv.txt', 't...",2026-10-29T21:18:20+00:00,colab-docstrange-v1,NaN
